# 同步编程

同步编程按顺序执行任务：前一个任务完成后，才会开始下一个。

下面的示例依次获取天气和新闻。由于每个请求都会阻塞等待，总耗时约等于两者之和（约 6 秒）。

```mermaid
flowchart TD
    Start([开始 main]) --> W1[开始获取天气]
    W1 --> W2[阻塞等待 4 秒<br/>time.sleep]
    W2 --> W3[天气数据返回]
    W3 --> N1[开始获取新闻]
    N1 --> N2[阻塞等待 2 秒<br/>time.sleep]
    N2 --> N3[新闻数据返回]
    N3 --> End([结束<br/>总耗时 ≈ 6 秒])

    classDef startEnd fill:#1B4F72,stroke:#0D2B45,color:#fff,stroke-width:2px
    classDef weather fill:#E74C3C,stroke:#922B21,color:#fff,stroke-width:2px
    classDef news fill:#27AE60,stroke:#1E8449,color:#fff,stroke-width:2px
    classDef wait fill:#F39C12,stroke:#B9770E,color:#fff,stroke-width:2px

    class Start,End startEnd
    class W1,W3 weather
    class N1,N3 news
    class W2,N2 wait
```

**时间线（串行，无法重叠）：**

```mermaid
gantt
    title 同步执行时间线
    dateFormat X
    axisFormat %s 秒
    section 天气
    获取天气 (阻塞 4s) :active, 0, 4
    section 新闻
    获取新闻 (阻塞 2s) :done, 4, 6
```

In [1]:
import time 

In [2]:

def fetch_weather():
    print("Fetching weather data...")
    time.sleep(4)  # Simulate a network delay
    print("Weather data fetched.")


def fetch_news():
    print("Fetching news data...")
    time.sleep(2)  # Simulate a network delay
    print("News data fetched.")



def main():
    start_time = time.time()

    fetch_weather()
    fetch_news()

    end_time = time.time()
    print(f"Total time taken: {end_time - start_time} seconds")




main()

Fetching weather data...
Weather data fetched.
Fetching news data...
News data fetched.
Total time taken: 6.00095534324646 seconds


# 异步编程

异步编程允许在等待 I/O（如网络请求）时切换执行其他任务，从而并发推进多个操作。

下面用 `asyncio.gather` 同时发起天气和新闻请求。总耗时接近最慢的那个任务（约 4 秒），而不是两者相加。

```mermaid
flowchart TD
    Start([开始 main]) --> Gather[asyncio.gather<br/>同时调度两个协程]
    Gather --> W1[协程: 获取天气]
    Gather --> N1[协程: 获取新闻]
    W1 --> W2[await sleep 4s<br/>让出事件循环]
    N1 --> N2[await sleep 2s<br/>让出事件循环]
    W2 --> Loop{事件循环<br/>切换任务}
    N2 --> Loop
    Loop --> N3[新闻先完成<br/>约 2s]
    Loop --> W3[天气后完成<br/>约 4s]
    N3 --> Done[两个任务都结束]
    W3 --> Done
    Done --> End([结束<br/>总耗时 ≈ 4 秒])

    classDef startEnd fill:#1B4F72,stroke:#0D2B45,color:#fff,stroke-width:2px
    classDef gather fill:#8E44AD,stroke:#5B2C6F,color:#fff,stroke-width:2px
    classDef weather fill:#E74C3C,stroke:#922B21,color:#fff,stroke-width:2px
    classDef news fill:#27AE60,stroke:#1E8449,color:#fff,stroke-width:2px
    classDef wait fill:#F39C12,stroke:#B9770E,color:#fff,stroke-width:2px
    classDef loop fill:#2980B9,stroke:#1A5276,color:#fff,stroke-width:2px

    class Start,End startEnd
    class Gather,Done gather
    class W1,W3 weather
    class N1,N3 news
    class W2,N2 wait
    class Loop loop
```

**时间线（并发，可重叠）：**

```mermaid
gantt
    title 异步执行时间线
    dateFormat X
    axisFormat %s 秒
    section 天气
    获取天气 await 4s :active, 0, 4
    section 新闻
    获取新闻 await 2s :done, 0, 2
    section 合计
    总耗时 ≈ max 而非 sum :crit, 0, 4
```

In [3]:
import asyncio
import time

In [5]:
async def fetch_weather():
    print("Fetching weather data...")
    await asyncio.sleep(4)  # Simulate a network delay
    print("Weather data fetched.")


async def fetch_news():
    print("Fetching news data...")
    await asyncio.sleep(2)  # Simulate a network delay
    print("News data fetched.")



async def main():
    start_time = time.time()

    await asyncio.gather(fetch_weather(), fetch_news())

    end_time = time.time()
    print(f"Total time taken: {end_time - start_time} seconds")



await main()

Fetching weather data...
Fetching news data...
News data fetched.
Weather data fetched.
Total time taken: 3.9940803050994873 seconds


In [ ]:
# 同步 vs 异步对比

```mermaid
flowchart LR
    subgraph Sync["同步：串行阻塞"]
        direction TB
        S1[天气 4s] --> S2[新闻 2s]
        S2 --> S3[合计 ≈ 6s]
    end

    subgraph Async["异步：并发等待"]
        direction TB
        A1[天气 4s]
        A2[新闻 2s]
        A1 --> A3[合计 ≈ 4s]
        A2 --> A3
    end

    classDef syncBox fill:#FADBD8,stroke:#E74C3C,color:#641E16,stroke-width:2px
    classDef asyncBox fill:#D5F5E3,stroke:#27AE60,color:#145A32,stroke-width:2px
    classDef slow fill:#E74C3C,stroke:#922B21,color:#fff,stroke-width:2px
    classDef fast fill:#27AE60,stroke:#1E8449,color:#fff,stroke-width:2px
    classDef total fill:#F39C12,stroke:#B9770E,color:#fff,stroke-width:2px

    class S1,S2 slow
    class S3 total
    class A1 slow
    class A2 fast
    class A3 total
```

| 对比项 | 同步 | 异步 |
|--------|------|------|
| 等待方式 | `time.sleep` 阻塞整个线程 | `await asyncio.sleep` 让出事件循环 |
| 调度方式 | 依次调用 | `asyncio.gather` 并发 |
| 总耗时 | ≈ 4 + 2 = **6 秒** | ≈ max(4, 2) = **4 秒** |
| 适用场景 | CPU 密集、逻辑简单 | I/O 密集（网络、磁盘、Agent 工具调用） |